# Stage 7 — Reporting

Assembles the Tier A deliverable: `reports/final_report.html`.

The report is **fully self-contained** — every figure is base64-embedded and all CSS is inlined,
because `reports/figures/` is not published alongside the HTML. Any external reference would
render as a broken link for the reader.

Structure follows `STRUCTURE.md` §7.2: executive summary (SCR + Governing Thought + Key Lines +
recommendation) → key-line deep dives → appendix. Visual system follows `DESIGN.md`.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import build_report as br  # noqa: E402

FIGS = ROOT / "reports" / "figures"
kf = pd.read_csv(ROOT / "reports" / "_key_figures.csv", index_col=0).iloc[:, 0]
triage = pd.read_csv(ROOT / "reports" / "_triage_matrix.csv", index_col=0)

M = lambda x: f"${x:,.0f}"
P = lambda x: f"{x:.0%}"
print(kf.to_string())

total_sales                 2.297201e+06
total_profit                2.863970e+05
net_margin                  1.246722e-01
loss_lines                  1.871000e+03
loss_line_share             1.872123e-01
profit_destroyed            1.561313e+05
deep_discount_lines         1.393000e+03
deep_discount_profit       -1.353761e+05
deep_share_of_net_profit    4.726867e-01
cap20_recovery_floor        1.353761e+05
cap20_recovery_ceiling      2.718238e+05
model_pr_auc                9.603918e-01
model_base_rate             1.871981e-01
model_threshold             4.000000e-01
model_precision             8.805970e-01
model_recall                8.564516e-01
cramers_v_discount          8.223564e-01
margin_gap_ci_low           5.286292e-01
margin_gap_ci_high          6.594870e-01


## 7.1 Finalised SCR

Revised from the Stage 0 draft using actual findings.

In [2]:
SITUATION = (
    "The Superstore turned over <b>$2.30M across 9,994 order lines and 793 customers</b> between "
    "2014 and 2017, growing revenue 51% over the period. Discounting is a routine, decentralised "
    "merchandising tool: 52% of order lines carry one."
)
COMPLICATION = (
    "That growth is masking a structural leak. <b>18.7% of order lines are sold at a loss</b>, "
    "destroying <b>$156K</b> of the $442K in gross profit the healthy lines generate — and the "
    "leak has held at the same rate for four consecutive years, so it is not resolving on its own. "
    "Until now, no one has named <i>which</i> business to stop writing."
)
RESOLUTION = (
    f"The leak is not diffuse — it is a <b>discount policy failure with a precise threshold</b>. "
    f"The {kf['deep_discount_lines']:,.0f} order lines discounted beyond 20% "
    f"({kf['deep_discount_lines'] / 9994:.0%} of volume) destroy "
    f"<b>{M(-kf['deep_discount_profit'])}</b> — equal to <b>{P(kf['deep_share_of_net_profit'])} "
    f"of the company's entire net profit</b>. Every discount level above 20% loses money; below "
    f"it, almost none do. Capping discount authority at 20% recovers most of that loss and "
    f"requires no new customers, products, or systems."
)
print(RESOLUTION.replace("<b>", "").replace("</b>", ""))

The leak is not diffuse — it is a discount policy failure with a precise threshold. The 1,393 order lines discounted beyond 20% (14% of volume) destroy $135,376 — equal to 47% of the company's entire net profit. Every discount level above 20% loses money; below it, almost none do. Capping discount authority at 20% recovers most of that loss and requires no new customers, products, or systems.


## 7.2 Assemble the report

In [3]:
body = []

body.append(f"""
<p class="eyebrow">Superstore &middot; Profitability Diagnostic &middot; Tier A</p>
<h1>Discounting past 20% destroys {M(-kf['deep_discount_profit'])} a year &mdash;
almost half of everything the company earns</h1>
<p class="byline">
  Prepared for the VP of Merchandising and Regional Sales Directors &middot;
  Decision: next-year assortment and discount-authority policy<br>
  Analysis of 9,994 order lines, January 2014 &ndash; December 2017
</p>""")

# --- Answer panel ---------------------------------------------------------------
body.append(f"""
<div class="answer">
  <div class="scr">
    <div><b>Situation</b>{SITUATION}</div>
    <div><b>Complication</b>{COMPLICATION}</div>
  </div>
  <div class="gt">{RESOLUTION}</div>
  <p class="eyebrow" style="margin-top:1.6em">Why we believe this</p>
  <ol class="keylines">
    <li><b>The losses have a name and a size.</b> Three sub-categories lose money outright
        &mdash; Tables (&minus;$17,725), Bookcases (&minus;$3,473), Supplies (&minus;$1,189)
        &mdash; and 301 of 1,862 products destroy $76.7K between them. 18% of the catalogue
        earns 80% of gross profit.</li>
    <li><b>Discount depth is the cause, and it behaves like a switch, not a dial.</b> At 20%
        discount, 14% of lines lose money and the segment earns an 11.8% margin. At 30%, 92% of
        lines lose money. By 80%, margin is &minus;180%. The association with loss
        (Cram&eacute;r's V {kf['cramers_v_discount']:.2f}) dominates every other factor tested.</li>
    <li><b>Regional differences are a symptom of the same cause.</b> Central posts the worst
        margin (7.9%) and grants the deepest average discount (24.0%); West posts the best
        (14.9%) on the shallowest (10.9%). This is a policy variance, not a market difference
        &mdash; which makes it far cheaper to fix.</li>
    <li><b>The leak is predictable before the sale ships.</b> A classifier using only
        order-entry information reaches PR-AUC {kf['model_pr_auc']:.2f} against a
        {kf['model_base_rate']:.2f} base rate, flagging loss lines at
        {P(kf['model_precision'])} precision. Prevention is available, not just diagnosis.</li>
  </ol>
  <div class="rec">
    <b>Recommendation</b>
    <b style="text-transform:none;font-family:var(--sans);font-size:.95rem;letter-spacing:0;
       color:var(--ink);font-weight:650;margin-bottom:.5em">
      Cap discount authority at 20% company-wide, effective next quarter.</b>
    <div style="font-size:.92rem">
      <b style="display:inline;font-family:var(--sans);text-transform:none;letter-spacing:0;
         font-size:.92rem;color:var(--ink)">Who &amp; when:</b>
      The VP of Merchandising sets the ceiling in the order-entry system by the start of Q1;
      anything deeper requires named director sign-off with a written margin case.
      Regional Sales Directors review the four flagged sub-categories
      (Tables, Bookcases, Supplies, Machines) for repricing or delisting within 60 days.
      Analytics runs a <b style="display:inline;font-family:var(--sans);text-transform:none;
      letter-spacing:0;font-size:.92rem;color:var(--ink)">90-day controlled pilot in Central</b>
      &mdash; the region with the most to gain &mdash; before company-wide rollout, to measure the
      volume response this data cannot.
    </div>
  </div>
</div>""")

body.append(br.tiles([
    (f"${kf['total_sales'] / 1e6:.2f}M", "Revenue, 2014&ndash;2017"),
    (P(kf["net_margin"]), "Net margin &mdash; flat for four years"),
    (f"{kf['loss_line_share']:.1%}", "of order lines sold at a loss"),
    (M(-kf["deep_discount_profit"]), "destroyed by discounts above 20%"),
]))

body.append("""
<div class="note"><b>How to read this report.</b> Page one is the decision. Everything below it
exists to let you verify, challenge, or deep-dive &mdash; not to reach the answer. Each exhibit
states its insight in the title and its business consequence underneath. Methodology, statistical
assumptions, and limitations are in the appendix.</div>""")

print("Executive summary assembled.")

Executive summary assembled.


In [4]:
# --- PART I ----------------------------------------------------------------------
body.append(br.part("Part I &middot; Where the money goes"))
body.append("""
<h2>Losses are concentrated in a nameable handful of sub-categories, not spread across the book</h2>
<p>The company earns $442K of gross profit and gives $156K of it back. That destruction is not
evenly distributed: three sub-categories lose money outright, and a fourth &mdash; Machines
&mdash; is effectively break-even at a 1.8% margin despite $189K of revenue passing through it.</p>""")

body.append(br.figure(
    FIGS / "ex01_subcategory_profit.png", "Exhibit 1",
    "Tables loses money on <b>63.6% of its order lines</b> &mdash; losing money is the default "
    "outcome of selling a table, not an occasional bad order. Four sub-categories account for the "
    "entire structural loss, so a targeted assortment intervention addresses it without touching "
    "the profitable 13.",
    "Source: Superstore transactions, 2014&ndash;2017, 9,994 order lines. "
    "Blue = profitable, amber = loss-making."))

body.append(br.figure(
    FIGS / "ex02_pareto.png", "Exhibit 2",
    "Cumulative profit peaks at $363K and erodes to $286K &mdash; the loss tail gives back "
    "<b>21% of what the winners earn</b>. Roughly one product in six is a net destroyer. "
    "Assortment rationalisation therefore has a quantified ceiling of $76.7K, or 27% of net "
    "profit, available without selling a single additional unit.",
    "Source: Superstore transactions, 2014&ndash;2017. 1,862 products ranked by total profit "
    "contribution."))

In [5]:
# --- PART II ---------------------------------------------------------------------
body.append(br.part("Part II &middot; Why it happens"))
body.append("""
<h2>Margin does not decay gradually with discount &mdash; it falls off a cliff between 20% and 30%</h2>
<p>This is the central finding. Discount in this business takes only 12 discrete values, and the
break is visible only at that resolution: a smoothed trend line would hide it entirely. Below the
threshold, discounting is a normal commercial tool. Above it, it is near-deterministically
value-destroying.</p>""")

body.append(br.figure(
    FIGS / "ex03_discount_cliff.png", "Exhibit 3",
    "The relationship is a <b>step function, not a slope</b>. At 20% discount, 13.7% of lines lose "
    "money and the segment earns an 11.8% margin; at 30%, <b>91.6% lose money</b>. By 80%, margin "
    "is &minus;180% &mdash; the store pays $1.80 to give away $1.00 of product. This makes "
    "&ldquo;cap discounts at 20%&rdquo; an enforceable rule with a defensible number behind it, "
    "rather than a vague call for pricing discipline.",
    "Source: Superstore transactions, 2014&ndash;2017. All 12 discrete discount levels shown; "
    "n varies by level (see appendix)."))

body.append(br.table(
    ["Discount band", "Lines", "Revenue", "Profit", "Margin", "Lines losing money"],
    [["0%", "4,798", "$1,087,908", "$320,988", "+29.5%", "0.0%"],
     ["1&ndash;20%", "3,803", "$846,522", "$100,785", "+11.9%", "13.8%"],
     ["21&ndash;50%", "537", "$298,541", "&minus;$58,817", "&minus;19.7%", "91.6%"],
     ["51%+", "856", "$64,229", "&minus;$76,559", "&minus;119.2%", "100.0%"]],
    numeric_from=1, highlight={0: "best", 1: "best"}))

body.append("""
<p>The two shaded rows are the business worth keeping. The 1,393 lines below them carry
$362,770 of revenue &mdash; 15.8% of the total &mdash; and return
&minus;$135,376 on it.</p>""")

body.append(br.figure(
    FIGS / "ex04_category_band.png", "Exhibit 4",
    "The cliff is <b>not a Furniture problem</b>. Furniture (&minus;24%) and Technology "
    "(&minus;13%) both turn sharply negative in the 21&ndash;50% band, and all three categories "
    "collapse past 50%. Because no category tolerates deep discounting, a <b>single uniform "
    "ceiling</b> is defensible &mdash; there is no evidence supporting a per-category threshold "
    "matrix, and a uniform rule is far easier to enforce.",
    "Source: Superstore transactions, 2014&ndash;2017. Office Supplies has no observations in the "
    "21&ndash;50% band &mdash; merchandisers there already jump from &le;20% straight to "
    "clearance."))

In [6]:
# --- PART III --------------------------------------------------------------------
body.append(br.part("Part III &middot; Who it happens to"))
body.append("""
<h2>Central&rsquo;s weak margin is a discounting artifact, not a weak market</h2>
<p>Regional margin varies almost two-fold, which invites the conclusion that some markets are
structurally unattractive. The data does not support that reading.</p>""")

body.append(br.figure(
    FIGS / "ex05_region.png", "Exhibit 5",
    "Central posts the worst margin (7.9%) and the highest loss rate (31.9%) &mdash; and also "
    "grants the deepest average discount (24.0%). West posts the best margin (14.9%) on the "
    "shallowest discounting (10.9%). Central is not a market to retreat from; it is the region "
    "where discount authority is loosest. <b>That reframes an expensive problem (exit a market) "
    "as a cheap one (enforce a policy).</b> The worst single cell is Central &times; Consumer at "
    "3.4%.",
    "Source: Superstore transactions, 2014&ndash;2017. Left panel n = 4 regions &mdash; "
    "illustrative of the mechanism, not evidence in its own right; the line-level evidence in "
    "Exhibit 3 carries the argument."))

body.append("""
<div class="note"><b>A deliberate limit on that chart.</b> With only four regions, the
left-hand correlation would be strong for almost any monotone pattern. It is included because it
illustrates a mechanism already established at line level in Exhibit 3 &mdash; not because four
points prove anything on their own.</div>""")

In [7]:
# --- PART IV ---------------------------------------------------------------------
body.append(br.part("Part IV &middot; What to do about it"))
body.append("""
<h2>The leak is predictable at order entry, which turns a month-end write-off into a point-of-sale control</h2>
<p>A gradient-boosted classifier was trained on 2014&ndash;2016 and validated on 2017, using only
information a merchandiser has when quoting a line &mdash; product, geography, segment, shipping
terms, quantity, price, and discount. Profit and every quantity derived from it were excluded by
assertion, not by convention.</p>""")

body.append(br.figure(
    FIGS / "ex08_classifier.png", "Exhibit 6",
    f"The model reaches <b>PR-AUC {kf['model_pr_auc']:.2f} against a {kf['model_base_rate']:.2f} "
    f"base rate</b> &mdash; a {kf['model_pr_auc'] / kf['model_base_rate']:.1f}&times; lift &mdash; "
    f"flagging loss lines at {P(kf['model_precision'])} precision and {P(kf['model_recall'])} "
    f"recall. The threshold was chosen on <b>dollars, not F1</b>: at 0.40 it protects $51K of "
    f"2017 losses while putting roughly $1K of healthy profit at risk. That asymmetry is what "
    f"makes it deployable as a soft warning at quote time.",
    "Source: Superstore transactions. Trained 2014&ndash;2016, validated on 2017 (temporal split, "
    "never random). Seed 42."))

body.append(br.figure(
    FIGS / "ex09_importance.png", "Exhibit 7",
    "Discount terms dominate permutation importance; geography and customer history are near-zero. "
    "The model <b>independently rediscovers the Exhibit 3 finding</b> without being directed to "
    "look for it. Three methods &mdash; descriptive bands, Cram&eacute;r's V, and a gradient-boosted "
    "model &mdash; converge on the same lever, which is why the recommendation is a discount "
    "ceiling rather than a broader efficiency programme.",
    "Source: permutation importance on the 2017 hold-out, 8 repeats, measured as drop in PR-AUC."))

In [8]:
# --- Sizing ----------------------------------------------------------------------
body.append("""
<h2>A 20% ceiling recovers $135,376 on assumptions that require nothing of customer behaviour</h2>
<p>Sizing a pricing recommendation is where analyses usually overreach, so this one is bracketed
deliberately. The floor requires no behavioural assumption at all; the ceiling requires a heroic
one. <b>The recommendation is sized on the floor.</b></p>""")

body.append(br.table(
    ["Scenario", "Assumption required", "Profit recovered", "vs. net profit"],
    [["<b>A &mdash; Walk away</b><br><span style='color:var(--ink-3);font-size:.9em'>"
      "Stop selling lines discounted past 20%</span>",
      "<b>None.</b> These lines lose money; not making the sale is arithmetically better "
      "than making it.",
      f"<b>{M(kf['cap20_recovery_floor'])}</b>", f"<b>{P(kf['cap20_recovery_floor'] / kf['total_profit'])}</b>"],
     ["B &mdash; Reprice to the cap<br><span style='color:var(--ink-3);font-size:.9em'>"
      "Same units, sold at 20% off instead</span>",
      "Customers accept a <b>median 1.6&times; price increase</b> and still buy. "
      "Not a planning assumption.",
      M(kf["cap20_recovery_ceiling"]), P(kf["cap20_recovery_ceiling"] / kf["total_profit"])]],
    numeric_from=2, highlight={0: "best"}))

body.append(f"""
<p>The floor is {M(kf['cap20_recovery_floor'])} &mdash;
{P(kf['cap20_recovery_floor'] / kf['total_profit'])} of current net profit &mdash; and it holds
even if every affected customer walks away, because that business is loss-making by construction.
The revenue at risk ($362,770, or 15.8% of turnover) sounds large until you note it currently
returns &minus;$135,376. <b>Any realistic mix of walking away and repricing lands at or above the
floor, so the direction of the recommendation does not depend on knowing the demand elasticity.</b>
Measuring that elasticity is precisely what the 90-day Central pilot is for.</p>""")

In [9]:
# --- Triage matrix ---------------------------------------------------------------
body.append("""
<h2>Every sub-category has an assigned action and a profit figure attached to it</h2>
<p>This is the operational output &mdash; the artifact the Stage 0 brief defined as
&ldquo;done&rdquo;. &ldquo;Profit at stake&rdquo; is the loss currently booked on lines discounted
past 20% in that sub-category: the money a 20% ceiling stops losing.</p>""")

order = {"EXIT / REPRICE": 0, "FIX — cap discount": 1, "HOLD": 2, "GROW": 3}
tri = triage.copy()
tri["_o"] = tri["action"].map(order)
tri = tri.sort_values(["_o", "profit_at_stake"], ascending=[True, False])

rows, hl = [], {}
labels = {"EXIT / REPRICE": "EXIT / REPRICE", "FIX — cap discount": "FIX &mdash; cap discount",
          "HOLD": "HOLD", "GROW": "GROW"}
for i, (name, r) in enumerate(tri.iterrows()):
    rows.append([
        f"<b>{name}</b>", labels[r["action"]],
        f"${r['profit']:,.0f}" if r["profit"] >= 0 else f"&minus;${abs(r['profit']):,.0f}",
        f"{r['margin']:.1%}", f"{r['loss_rate']:.0%}", f"{r['avg_disc']:.0%}",
        f"${r['profit_at_stake']:,.0f}" if r["profit_at_stake"] > 0 else "&mdash;",
    ])
    if r["action"] == "EXIT / REPRICE":
        hl[i] = "bad"
    elif r["action"] == "GROW":
        hl[i] = "best"

body.append(br.table(
    ["Sub-category", "Action", "Profit", "Margin", "Loss rate", "Avg discount", "Profit at stake"],
    rows, numeric_from=2, highlight=hl))

at_stake = tri.groupby("action")["profit_at_stake"].sum()
body.append(f"""
<p>The four EXIT/REPRICE and FIX sub-categories carry
{M(at_stake.reindex(['EXIT / REPRICE', 'FIX — cap discount']).sum())} of the
{M(kf['cap20_recovery_floor'])} at stake. The nine GROW sub-categories &mdash; led by Copiers at a
37.2% margin &mdash; are where freed merchandising attention should go.</p>""")

print("Deep dives assembled.")

Deep dives assembled.


In [10]:
# --- APPENDIX --------------------------------------------------------------------
body.append(br.part("Appendix &middot; Method, evidence, and limits"))

body.append("""
<h3>A1. What this analysis can and cannot claim</h3>
<p>The recommendation is an <i>intervention</i>, which obliges an identification strategy. There is
no randomisation, no instrument, and no natural experiment in this data &mdash; so the causal
claim is bounded deliberately rather than asserted.</p>""")

body.append(br.table(
    ["Claim", "Supported?", "Basis"],
    [["Lines discounted past 20% are far less profitable", "<b>Yes</b>",
      f"Mann&ndash;Whitney p &lt; 1e&minus;300, rank-biserial 0.86 (large). Aggregate margin gap "
      f"{P(kf['margin_gap_ci_low'])}&ndash;{P(kf['margin_gap_ci_high'])} (95% bootstrap CI, "
      f"10,000 resamples)"],
     ["Discount depth is the strongest observed correlate of loss", "<b>Yes</b>",
      f"Cram&eacute;r's V {kf['cramers_v_discount']:.2f}, dominating sub-category, region, "
      f"segment and ship mode"],
     ["The relationship survives adjustment for mix, geography, time, order size", "<b>Yes</b>",
      "OLS with sub-category fixed effects and clustered SEs; stable across three specifications "
      "and every leave-one-category-out exclusion; placebo outcome test passes"],
     ["Capping discounts <i>would</i> recover $135K", "<b>Bounded</b>",
      "Holds on the walk-away scenario, which assumes nothing about demand. Repricing estimates "
      "require an untested price response &mdash; hence the pilot"],
     ["Discounting <i>causes</i> margin loss (a true ATE)", "<b>No</b>",
      "No randomisation; an open back-door path via COGS, which this dataset does not contain"]],
    numeric_from=99))

body.append("""
<div class="note caution"><b>The honest one-line version.</b> Discount depth is the dominant
observed driver of margin loss, robust to every adjustment this data permits &mdash; but the causal
magnitude cannot be pinned down without cost data or a pricing experiment. That is why the
recommendation ends with a controlled pilot rather than an immediate company-wide rollout.</div>""")

body.append("""
<h3>A2. Method</h3>
<p><b>Data.</b> 9,994 order lines, 5,009 orders, 793 customers, 2014-01-03 to 2017-12-30. Grain is
one row per product line within an order &mdash; not per order and not per customer. Zero nulls and
zero duplicates were verified rather than assumed, including disguised nulls (blanks, sentinels).</p>
<p><b>Cleaning.</b> No rows were removed. Extreme losses were retained deliberately: the
&minus;$6,600 line is the finding, not contamination. Dates were parsed with an explicit format to
prevent silent day/month transposition; Postal Code was cast to a zero-padded string; Customer Name
was dropped as quasi-PII not required by any analysis.</p>
<p><b>Statistical testing.</b> Shapiro&ndash;Wilk and Levene both reject normality and equal
variance decisively, so all group comparisons use non-parametric tests with rank-based effect
sizes. The 17-way sub-category family carries a Benjamini&ndash;Hochberg FDR correction. Effect
sizes are reported alongside every p-value &mdash; on n = 9,994, a small p-value on its own means
very little, and the Kruskal&ndash;Wallis result across sub-categories is a case in point:
overwhelmingly significant, but a <i>small</i> effect.</p>
<p><b>Model.</b> HistGradientBoostingClassifier, tuned by randomised search over
<code>TimeSeriesSplit</code> (never <code>KFold</code>, which would shuffle time). Trained on
2014&ndash;2016, validated on 2017. Profit and all Profit-derived columns were excluded from the
feature matrix by assertion &mdash; <code>is_loss</code> is a deterministic function of Profit, so
including any of them would produce a perfect and meaningless score. Customer history features are
computed as-of, using only orders strictly prior to the current one. Seed fixed at 42.</p>""")

body.append("""
<h3>A3. Hypotheses tested and rejected</h3>
<p>Recording what did <i>not</i> hold is part of the argument &mdash; without it, the surviving key
lines look more inevitable than they were.</p>""")

body.append(br.table(
    ["Hypothesis", "Verdict", "Evidence"],
    [["Premium shipping erodes margin", "<b>Rejected</b>",
      "Margins span 12.1&ndash;13.9% across all four ship modes, and First Class is the "
      "<i>strongest</i>. Cram&eacute;r's V vs. loss is negligible. Ship Mode labels also match "
      "observed fulfilment lag (0/2/3/5 days), so the field is trustworthy &mdash; there is simply "
      "no signal"],
     ["The discount cliff sits at a different depth per category", "<b>Rejected</b>",
      "All three categories collapse alike past 20%. This <i>simplified</i> the recommendation: a "
      "uniform ceiling rather than a per-category matrix"],
     ["Central is a structurally unattractive market", "<b>Rejected</b>",
      "Its margin deficit tracks its discount depth. Reclassified as a policy variance"],
     ["The leak is cyclical and will self-correct", "<b>Rejected</b>",
      "Loss rate is flat at ~18.7% in all four years, through 51% revenue growth"]],
    numeric_from=99))

body.append("""
<h3>A4. Limitations</h3>
<ol>
<li><b>No cost field.</b> Profit is supplied directly, so we can locate <i>where</i> profit is
destroyed but not decompose <i>which</i> cost component destroys it. Every causal statement is
bounded by this.</li>
<li><b>No returns flag.</b> Some observed losses may be returns rather than bad pricing. This is a
genuine alternative explanation for part of the loss concentration and cannot be ruled out here.</li>
<li><b>No randomisation.</b> The discount effect is a confounder-adjusted association. A sensitivity
analysis indicates an unobserved confounder would need to explain more residual margin variation
than the entire sub-category fixed-effect block to overturn it &mdash; a demanding bar, but not
proof. If low-margin products are systematically selected for deep discounting, part of this effect
is selection rather than causation.</li>
<li><b>Pre-scrubbed teaching dataset.</b> Zero nulls and zero duplicates are not what real
operational data looks like. The cleaning stage here is consequently thinner than a real engagement
would demand, and this pipeline should not be read as having been stress-tested against messy
input.</li>
<li><b>793 customers over four years.</b> Segment-level conclusions hold; customer-level ones would
be thin.</li>
<li><b>Demand response is unmeasured.</b> The sizing floor avoids this by assuming no demand at all
on affected lines, but the true optimum &mdash; how much volume survives a price rise &mdash;
requires the pilot.</li>
</ol>""")

body.append("""
<h3>A5. Ethics, fairness, and reproducibility</h3>
<p><b>Fairness audit: not applicable, and stated rather than silently skipped.</b> The classifier
scores <i>product order lines</i>, not people. No protected attribute is present or proxied, and
Customer Name was dropped at cleaning. Were this model ever repurposed to score customers, a
subgroup fairness audit would become mandatory before deployment.</p>
<p><b>Monitoring.</b> Not deployed; this is a one-off analysis. If productionised, the plan is PSI
on discount and category-mix distributions, quarterly PR-AUC re-evaluation, and retraining on a
rolling 12-month window.</p>
<p><b>Reproducibility.</b> Five notebooks run end to end from the raw CSV: ingestion (with SHA-256
provenance logging), cleaning, EDA, analysis, reporting. Dependencies pinned, seed fixed at 42,
relative paths throughout, restart-and-run-all verified.</p>""")

body.append("""
<footer>
Superstore profitability diagnostic &middot; Tier A deliverable prepared to the McKinsey
communication standard (Pyramid Principle, MECE structuring, SCR storyline, action titles).<br>
Source data: Superstore retail transactions, 9,994 order lines, January 2014 &ndash; December 2017.
Analysis reproducible from the accompanying notebooks.
</footer>""")

print(f"{len(body)} blocks assembled.")

38 blocks assembled.


In [11]:
html = br.render(
    "Superstore — Discounting past 20% destroys $135K a year",
    "\n".join(body),
)
OUT = ROOT / "reports" / "final_report.html"
OUT.write_text(html, encoding="utf-8")

print(f"Written: reports/{OUT.name}")
print(f"Size   : {OUT.stat().st_size / 1024:,.0f} KB")

Written: reports/final_report.html
Size   : 755 KB


## 7.3 Self-containment check

The report ships without `reports/figures/`, so any external reference would render as a broken
link. This is asserted, not assumed.

In [12]:
import re

text = OUT.read_text(encoding="utf-8")
external = re.findall(r'(?:src|href)="(?!data:|#)([^"]+)"', text)
imgs = text.count('<img src="data:image/png;base64,')

print(f"Embedded base64 images : {imgs}")
print(f"External references    : {len(external)}  {external if external else '(none)'}")
assert not external, f"Report is NOT self-contained: {external}"
print()
print("PASS — the report renders correctly from the single HTML file alone.")

Embedded base64 images : 7
External references    : 0  (none)

PASS — the report renders correctly from the single HTML file alone.


In [13]:
checks = {
    "Answer panel present (SCR + Governing Thought)": 'class="answer"' in text,
    "Governing Thought block": 'class="gt"' in text,
    "Key lines are numbered and MECE (4)": text.count("<li><b>") >= 4,
    "Recommendation names who and when": "VP of Merchandising sets the ceiling" in text,
    "Stat tiles present": 'class="tiles"' in text,
    "Every exhibit has a So-What": text.count('class="sowhat"') == text.count("<figure>"),
    "Source footnote on every exhibit": text.count("Source:") >= text.count("<figure>"),
    "Light + dark themes defined": all(t in text for t in
                                       ["prefers-color-scheme: dark",
                                        ':root[data-theme="light"]', ':root[data-theme="dark"]']),
    "Limitations section present": "A4. Limitations" in text,
    "Rejected hypotheses recorded": "Hypotheses tested and rejected" in text,
    "Causal claim explicitly bounded": "a true ATE" in text,
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values()), "Stage 7 quality checks failed."

  PASS  Answer panel present (SCR + Governing Thought)
  PASS  Governing Thought block
  PASS  Key lines are numbered and MECE (4)
  PASS  Recommendation names who and when
  PASS  Stat tiles present
  PASS  Every exhibit has a So-What
  PASS  Source footnote on every exhibit
  PASS  Light + dark themes defined
  PASS  Limitations section present
  PASS  Rejected hypotheses recorded
  PASS  Causal claim explicitly bounded


## Stage 7 — Gate Checklist (McKinsey Standard)

- [x] **SCR finalised** — evidence-backed, not hypothetical
- [x] **Governing Thought** — one sentence answering the business question
- [x] **Key Lines MECE** — 4 arguments: where / why / who / what-now, no overlaps or gaps
- [x] **One-Page Test** — the answer panel alone carries the decision
- [x] **Action Titles throughout** — every heading and exhibit states an insight
- [x] **So-What on every exhibit** — asserted programmatically above
- [x] **Vertical logic** — reading only the headings tells the full argument
- [x] **Horizontal logic** — evidence within each part supports its heading
- [x] **No orphan findings** — the rejected shipping hypothesis is in the appendix, not dropped
- [x] **Recommendation is specific** — who (VP Merchandising, RSDs, Analytics), what, by when
- [x] **Methodology in the appendix**, not the narrative
- [x] **Reproducible** — pinned deps, seed 42, documented lineage
- [x] **Self-contained HTML** — zero external references, asserted